In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

mambo = pd.read_csv("base/mambo.csv")
marche = pd.read_csv("base/marche.csv")
base_cafe = pd.read_csv("base/base_cafe.csv")

In [ ]:
mambo.head()

In [ ]:
marche.head()

In [ ]:
base_cafe.head()

In [ ]:
base_cafe[" Marca"]

In [ ]:
base_cafe["Marca"][base_cafe["Marca"].duplicated()]

In [ ]:
contagem = base_cafe["Marca"].value_counts()

contagem[contagem > 1]

In [ ]:
base_cafe[base_cafe["Marca"] == "pa"]

In [ ]:
mambo.shape
marche.shape
base_cafe.shape

In [ ]:
base_total = pd.concat([base_cafe, mambo, marche], ignore_index=True)

In [ ]:
base_total[base_total["Loja"] == "Mambo"]

In [ ]:
base_recente = (
    base_cafe
    .sort_values("Data")
    .groupby("Código")
    .last()
    .reset_index()
)

In [ ]:
base_recente.groupby("Marca")["Média Preço Normal"].mean().sort_values()

In [ ]:
base_recente["Marca"].value_counts()

In [ ]:
base_recente["Média Preço Normal"].hist(bins=20)

plt.title("Distribuição de preço do café")
plt.xlabel("Preco")
plt.ylabel("Quantidade")

plt.show()

In [ ]:
base_recente.groupby("Website Monitorado")["Média Preço Normal"].mean()

In [ ]:
base_recente = base_recente.rename(columns={
    "Descrição": "Titulo",
    "Marca": "Fabricante",
    "Média Preço Normal": "Preco",
    "Website Monitorado": "Loja"
})

print(base_recente)

In [ ]:
mambo = mambo.rename(columns={
    "Titulo": "Titulo",
    "Fabricante": "Fabricante",
    "Preco": "Preco"
})

mambo["Loja"] = "Mambo"

mambo = mambo[["Titulo","Fabricante","Preco","Loja"]]
print(mambo)

In [ ]:
marche["Loja"] = "St Marche"

marche = marche[["Titulo","Fabricante","Preco","Loja"]]

base_recente = base_recente[["Titulo","Fabricante","Preco","Loja"]]

In [ ]:
base_total = pd.concat([base_recente, mambo, marche], ignore_index=True)

In [ ]:
base_total.groupby("Fabricante")["Preco"].mean()

In [ ]:
pd.crosstab(base_total["Fabricante"], base_total["Loja"])

In [ ]:
base_total.pivot_table(
    index="Titulo",
    columns="Loja",
    values="Preco"
)

In [ ]:
comparacao = base_total.pivot_table(
    index="Titulo",
    columns="Loja",
    values="Preco"
)

comparacao.head(20)

In [ ]:
base_total.groupby("Loja")["Titulo"].count()

In [ ]:
base_total.groupby("Loja")["Preco"].mean().sort_values(ascending=False)

A análise de preços de café entre os supermercados
indica diferenças claras de posicionamento. O St Marche
apresenta o maior preço médio R$36 e mediana de R$31,
sugerindo presença significativa de produtos premium.

In [ ]:
base_total.groupby("Loja")["Preco"].median().sort_values()

- St Marche é o supermercado mais premium

- Pão de Açúcar tem preços mais competitivos

In [ ]:
analise_loja = (
    base_total
    .groupby("Loja")["Preco"]
    .agg(["mean","median","min","max","count"])
    .reset_index()
)

analise_loja

In [ ]:
#analise_loja.to_csv("results/analise_preco_loja.csv", index=False)

In [ ]:
analise_fabricante = (
    base_total.groupby("Fabricante")["Preco"]
    .agg(["mean","median","min","max","count"])
    .sort_values("mean")
    .reset_index()
)

# Calcular range
analise_fabricante["range"] = (analise_fabricante["max"] - analise_fabricante["min"]).round(2)

# Arredondar
analise_fabricante[["mean","median","min","max"]] = analise_fabricante[["mean","median","min","max"]].round(2)

In [ ]:
#analise_fabricante.to_csv("results/analise_preco_fabricante.csv", index=False)

In [ ]:
analise_mix = pd.crosstab(
    base_total["Fabricante"], base_total["Loja"]
)

analise_mix

In [ ]:
#analise_mix.to_csv("results/analise_mix.csv")

In [ ]:
analise_preco = base_total["Preco"].describe().round(2)

analise_preco

In [ ]:
#analise_preco.to_csv("results/analise_distribuicao_preco.csv")

In [ ]:
bins = [0,20,30,40,50,100]

base_total["Faixa_Preco"] = pd.cut(base_total["Preco"], bins=bins)

base_total["Faixa_Preco"].value_counts().sort_index(ascending=True)

In [ ]:
top_marcas = (
    base_total.groupby("Fabricante")["Preco"]
    .mean()
    .sort_values(ascending=False)
    .head(5)
)

top_marcas.plot(kind="bar")

In [ ]:
top_marcas = (
    base_total.groupby("Fabricante")["Preco"]
    .mean()
    .sort_values(ascending=True)
    .head(5)
)

top_marcas.plot(kind="bar")

In [ ]:
top = base_total["Fabricante"].value_counts().head(5).index

base_total["marca_grupo"] = base_total["Fabricante"].apply(
    lambda x: x if x in top else "Outros"
)

sns.histplot(base_total["Preco"], bins=30);

In [ ]:
plt.figure(figsize=(12, 6))

sns.kdeplot(base_total["Preco"], fill=True, bw_adjust=0.7, color="skyblue")

plt.axvline(base_total["Preco"].median(), color="red", linestyle="solid", label="Mediana")
plt.axvline(base_total["Preco"].mean(), color="red", linestyle="dotted", label="Média")

plt.title("Distribuição de preços KDE")
plt.xlabel("Preço")
plt.xticks(np.arange(0, 210, 10))
plt.ylabel("Densidade")
plt.legend()
plt.grid(alpha=0.3)

plt.show()


In [ ]:
media = base_total['Preco'].mean()
std = base_total['Preco'].std()

# Produtos dentro da faixa ±1 std
dentro_1std = base_total[(base_total['Preco'] >= media - std) & (base_total['Preco'] <= media + std)]

# Porcentagem real
percent_dentro_1std = len(dentro_1std) / len(base_total) * 100
print(f'Percentual real dentro de ±1 std: {percent_dentro_1std:.2f}%')

A maior parte dos produtos está concentrada na faixa ±1 std.

Os outliers (muito baratos ou caros) distorcem a média, mas são poucos.

A mediana é mais representativa do preço típico da maioria dos produtos.

In [ ]:
media = base_total['Preco'].mean()
mediana = base_total['Preco'].median()
std = base_total['Preco'].std()

lower = media - std
upper = media + std

dentro_1std = base_total[(base_total['Preco'] >= lower) & (base_total['Preco'] <= upper)]
percent_dentro_1std = len(dentro_1std) / len(base_total) * 100

plt.figure(figsize=(14,6))

# Curva KDE
sns.kdeplot(base_total['Preco'], fill=True, bw_adjust=0.7, color='#1f77b4', alpha=0.5)

# Faixas de preço
plt.axvspan(0, 10, color='#2ca02c', alpha=0.25, label='Barato (<R$10)')
plt.axvspan(10, 38, color='#17becf', alpha=0.25, label='Médio (R$10–38)')
plt.axvspan(38, base_total['Preco'].max(), color='#9467bd', alpha=0.25, label='Premium (>R$38)')

# Média e mediana
plt.axvline(media, color='#d62728', linestyle='--', linewidth=2, label=f'Média: {media:.2f}')
plt.axvline(mediana, color='black', linestyle='--', linewidth=2, label=f'Mediana: {mediana:.2f}')

# ±1 std
plt.axvline(lower, color='#ffbf00', linestyle='--', linewidth=1.5)
plt.axvline(upper, color='#ffbf00', linestyle='--', linewidth=1.5)

plt.fill_betweenx(
    y=[0, plt.gca().get_ylim()[1]],
    x1=lower,
    x2=upper,
    color='#ffbf00',
    alpha=0.15,
    label=f'±1 std ≈ {percent_dentro_1std:.2f}%'
)

plt.title('Distribuição de Preços e ±1 Desvio Padrão', fontsize=16)
plt.xlabel('Preço (R$)', fontsize=14)
plt.xticks(np.arange(0, 210, 10))
plt.ylabel('Densidade', fontsize=14)

plt.legend()
plt.grid(alpha=0.3)

plt.show()

A maior parte dos produtos está concentrada em preços médios, o que indica que a estratégia de preço e estoque deve priorizar essa faixa.

A faixa ±1 std cobre praticamente toda a distribuição central (85,74%), mostrando onde o consumidor típico encontra produtos.

Produtos fora dessa faixa são exceções, que podem ser usados para promoções, ofertas de entrada ou produtos premium.

Média > mediana → indica assimetria à direita, causada por produtos caros, então a mediana reflete melhor o preço típico do catálogo.